In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from PIL import Image
from sklearn.metrics import auc
import shutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_DIR, DATASET_ZIP, MODELS_DIR, MAPS_DIR, MAPS_ZIP, CONFIGS, LABELS, LABEL_ID, IMAGENET_MEAN, IMAGENET_STD, MODELS
from utils import build_model, apply_nclahe, get_stratification, saliency_entropy, compute_mmd

In [ ]:
THRESHOLD_ACTIVATION = 0.2 # Threshold for binarizing activation maps when computing IoU.

In [ ]:
# Decompress dataset and saliency maps
print('Decompressing dataset and saliency maps...')
os.system(f'unzip -q {DATASET_ZIP} -d {PROJECT_DIR}')
os.system(f'unzip -q {MAPS_ZIP} -d {PROJECT_DIR}')
print('All done!')

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
# Loading predictions by image
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')
print(f'Predictions loaded: {len(df_preds)} rows')
print(df_preds.head())

In [ ]:
# Saliency metrics
def pointing_game(cam, x_min, y_min, x_max, y_max):
    """1 if the maximum activation pixel falls within the BB, 0 otherwise."""
    max_idx = np.unravel_index(cam.argmax(), cam.shape)
    py, px = max_idx
    return int(x_min <= px <= x_max and y_min <= py <= y_max)


def poe(cam, x_min, y_min, x_max, y_max):
    """Proportion of energy of the map within the BB."""
    total = cam.sum()
    if total == 0:
        return 0.0
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    dentro = cam[y_min:y_max, x_min:x_max].sum()
    return float(dentro / total)


# Heatmap/pathological anatomy convergence metrics - BB (requires binarization of the saliency map)

def binarize_map(map, threshold=0.20):
    """Binarizes the heatmap with a threshold over the maximum value."""
    return (map >= threshold * map.max()).astype(np.float32)


def dsc(cam_bin, x_min, y_min, x_max, y_max, res):
    """Dice Similarity Coefficient between binarized map and BB mask."""
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    mask = np.zeros((res, res), dtype=np.float32)
    mask[y_min:y_max, x_min:x_max] = 1.0
    intersection = (cam_bin * mask).sum()
    suma = cam_bin.sum() + mask.sum()
    if suma == 0:
        return 0.0
    return float(2 * intersection / suma)


def iou_between_maps(cam1, cam2, threshold=0.20):
    """IoU between two binarized heatmaps (discrimination between labels)."""
    bin1 = binarize_map(cam1, threshold)
    bin2 = binarize_map(cam2, threshold)
    intersection = (bin1 * bin2).sum()
    union = ((bin1 + bin2) > 0).sum()
    if union == 0:
        return 0.0
    return float(intersection / union)

In [ ]:
# Deletion AUC and Insertion AUC

def deletion_insertion_auc(model, img_tensor, cam, class_idx, n_steps=10):
    """
    Calculates Deletion AUC and Insertion AUC.
     - Deletion: progressively removes the most important pixels
     - Insertion: progressively introduces the most important pixels
    Returns:
      (deletion_auc, insertion_auc)
    """
    model.eval()
    img_np = img_tensor.squeeze().cpu().numpy()  # (3, H, W)
    H, W = img_np.shape[1], img_np.shape[2]

    # Sort pixels by importance (highest to lowest)
    cam_flat = cam.flatten()
    sort = np.argsort(cam_flat)[::-1]

    # Reference image for insetion (blurred image)
    blur_img = cv2.GaussianBlur(
        img_np.transpose(1, 2, 0),
        (51, 51), 0
    ).transpose(2, 0, 1)

    deletion_scores = []
    insertion_scores = []

    n_pixels = H * W
    steps = [int(n_pixels * i / n_steps) for i in range(n_steps + 1)]

    img_del = img_np.copy().reshape(3, -1)
    img_ins = blur_img.copy().reshape(3, -1)

    for step in steps:
        # Deletion: put to zero the most important pixels
        img_del_step = img_np.copy().reshape(3, -1)
        img_del_step[:, sort[:step]] = 0
        tensor_del = torch.FloatTensor(img_del_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        # Insertion: reveal the most important pixels from the blurred image
        img_ins_step = blur_img.copy().reshape(3, -1)
        img_ins_step[:, sort[:step]] = img_np.reshape(3, -1)[:, sort[:step]]
        tensor_ins = torch.FloatTensor(img_ins_step.reshape(3, H, W)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            prob_del = torch.sigmoid(model(tensor_del))[0, class_idx].item()
            prob_ins = torch.sigmoid(model(tensor_ins))[0, class_idx].item()

        deletion_scores.append(prob_del)
        insertion_scores.append(prob_ins)

    x = np.linspace(0, 1, n_steps + 1)
    del_auc = auc(x, deletion_scores)
    ins_auc = auc(x, insertion_scores)

    return del_auc, ins_auc

In [ ]:
class PenultimateFeatureExtractor:
    """Captures the penultimate-layer output for MMD (section 5.2).
    AlexNet: classifier[5] (ReLU before the final fc). DenseNet-121: the
    pooled feature vector feeding `classifier`, captured via a pre-hook."""
    def __init__(self, model, architecture):
        self.features = None
        self.architecture = architecture
        if architecture == 'alexnet':
            model.classifier[5].register_forward_hook(self._hook)
        elif architecture == 'densenet':
            model.classifier.register_forward_pre_hook(self._pre_hook)

    def _hook(self, module, input, output):
        self.features = output.detach().cpu().numpy()

    def _pre_hook(self, module, input):
        self.features = input[0].detach().cpu().numpy()

    def get(self):
        return self.features.squeeze(0)

In [ ]:
# Main

registers = []
features_by_group = {}  # {(model, label): {'TP': [...], 'FP': [...], 'FN': [...], 'TN': [...], 'pathology': [...], 'control': [...]}}

for cfg in CONFIGS:
    name = cfg['name']
    res = cfg['res']
    print(f"\n{'='*50}")
    print(f"Calculating metrics for {name}...")

    df_meta_full = pd.read_csv(cfg['metadata'])
    df_meta = df_meta_full[df_meta_full['class_id'].isin([0, 1])]
    df_pred_model = df_preds[df_preds['model'] == name]

    model = build_model(cfg['arq'], cfg['res'])
    pth = f"{MODELS_DIR}/{cfg['arq']}_{res}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()
    for module in model.modules():
        if isinstance(module, torch.nn.ReLU):
            module.inplace = False

    feature_extractor = PenultimateFeatureExtractor(model, cfg['arq'])

    normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    maps_dir = f'{MAPS_DIR}/{name}'
    image_ids = df_pred_model['image_id'].unique()

    for i, image_id in enumerate(image_ids):
        npz_path = f'{maps_dir}/{image_id}.npz'
        if not os.path.exists(npz_path):
            continue
        maps = np.load(npz_path)

        img_path = f'{DATASET_DIR}/images_256/{image_id}.png' if res == 256 else f'{DATASET_DIR}/images_1024/{image_id}.png'
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f'Not found: {img_path}')
            continue

        tile_size = 4 if res == 256 else 16
        img_clahe = apply_nclahe(img, tile_size=tile_size)
        img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
        img_tensor = normalize(transforms.ToTensor()(Image.fromarray(img_rgb))).unsqueeze(0)

        pred_row = df_pred_model[df_pred_model['image_id'] == image_id]
        if len(pred_row) == 0:
            continue
        pred_row = pred_row.iloc[0]

        with torch.no_grad():
            model(img_tensor.to(DEVICE))
        features_vec = feature_extractor.get()

        is_control = (pred_row['label_aneurysm'] == 0) and (pred_row['label_cardiomegaly'] == 0)

        for label_name in LABELS:
            class_idx = LABEL_ID[label_name]
            cam_key = f'gradcam_{label_name}'
            if cam_key not in maps:
                continue
            cam = maps[cam_key]

            label_col, pred_col = f'label_{label_name}', f'pred_{label_name}'
            label = int(pred_row[label_col])
            pred = int(pred_row[pred_col])

            if label == 1 and pred == 1: stratification = 'TP'
            elif label == 0 and pred == 1: stratification = 'FP'
            elif label == 1 and pred == 0: stratification = 'FN'
            else: stratification = 'TN'

            bb_rows = df_meta[(df_meta['image_id'] == image_id) & (df_meta['class_id'] == class_idx)]
            has_bb = len(bb_rows) > 0
            pg = poe_val = dsc_val = np.nan
            if has_bb:
                bb = bb_rows.iloc[0]
                pg = pointing_game(cam, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'])
                poe_val = poe(cam, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'])
                cam_bin = binarize_map(cam)
                dsc_val = dsc(cam_bin, bb['x_min'], bb['y_min'], bb['x_max'], bb['y_max'], res)

            iou_maps = np.nan
            cam_other_key = 'gradcam_cardiomegaly' if label_name == 'aneurysm' else 'gradcam_aneurysm'
            if cam_other_key in maps.files:
                iou_maps = iou_between_maps(cam, maps[cam_other_key])

            del_auc, ins_auc = deletion_insertion_auc(model, img_tensor, cam, class_idx, n_steps=10)
            entropy_val = saliency_entropy(cam)

            group_key = (name, label_name)
            features_by_group.setdefault(group_key, {'TP': [], 'FP': [], 'FN': [], 'TN': [], 'pathology': [], 'control': []})
            features_by_group[group_key][stratification].append(features_vec)
            if label == 1:
                features_by_group[group_key]['pathology'].append(features_vec)
            if is_control:
                features_by_group[group_key]['control'].append(features_vec)

            agg_row = df_agg_metrics[(df_agg_metrics['Model'] == name) & (df_agg_metrics['Label'] == label_name)]
            mcc_agg = agg_row['MCC'].values[0] if len(agg_row) else np.nan
            auc_roc_agg = agg_row['AUC-ROC'].values[0] if len(agg_row) else np.nan
            pr_auc_agg = agg_row['PR-AUC'].values[0] if len(agg_row) else np.nan
            sens_agg = agg_row['Recall'].values[0] if len(agg_row) else np.nan
            spec_agg = agg_row['Specificity'].values[0] if len(agg_row) else np.nan

            threshold_mcc = pred_row.get(f'threshold_mcc_{label_name}', np.nan)
            threshold_max_sens = pred_row.get('threshold_max_sens_aneurysm', np.nan) if label_name == 'aneurysm' else np.nan

            registers.append({
                'image_id': image_id, 'model': name, 'label': label_name, 'resolution': res, 'iteracion': 1,
                'patient_age': pred_row.get('patient_age', np.nan),
                'patient_age_group': pred_row.get('patient_age_group', np.nan),
                'patient_sex': pred_row.get('patient_sex', np.nan),
                'verdict': stratification,
                'threshold_mcc': threshold_mcc, 'threshold_max_sens': threshold_max_sens,
                'MCC': mcc_agg, 'AUC_ROC': auc_roc_agg, 'PR_AUC': pr_auc_agg,
                'sensitivity': sens_agg, 'specificity': spec_agg,
                'PG': pg, 'PoE': poe_val, 'DSC': dsc_val, 'IoU_between_maps': iou_maps,
                'Del_AUC': del_auc, 'Ins_AUC': ins_auc, 'entropy_gradcam': entropy_val,
            })

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(image_ids)} processed')

    print(f'  {name} completed.')

df_metrics = pd.DataFrame(registers)
print(f'\n Computed per-image-per-label rows: {len(df_metrics)}')

In [ ]:
MIN_GROUP_SIZE = 2
mmd_results = {}

for (model_name, label_name), groups in features_by_group.items():
    result = {'pathology_vs_control': np.nan, 'TP_vs_FP': np.nan, 'TP_vs_FN': np.nan}
    if len(groups['pathology']) >= MIN_GROUP_SIZE and len(groups['control']) >= MIN_GROUP_SIZE:
        result['pathology_vs_control'] = compute_mmd(np.array(groups['pathology']), np.array(groups['control']))
    if len(groups['TP']) >= MIN_GROUP_SIZE and len(groups['FP']) >= MIN_GROUP_SIZE:
        result['TP_vs_FP'] = compute_mmd(np.array(groups['TP']), np.array(groups['FP']))
    if len(groups['TP']) >= MIN_GROUP_SIZE and len(groups['FN']) >= MIN_GROUP_SIZE:
        result['TP_vs_FN'] = compute_mmd(np.array(groups['TP']), np.array(groups['FN']))
    mmd_results[(model_name, label_name)] = result

mmd_cross_label = {}
for model_name in MODELS:
    feats_aneu = features_by_group.get((model_name, 'aneurysm'), {}).get('pathology', [])
    feats_card = features_by_group.get((model_name, 'cardiomegaly'), {}).get('pathology', [])
    if len(feats_aneu) >= MIN_GROUP_SIZE and len(feats_card) >= MIN_GROUP_SIZE:
        mmd_cross_label[model_name] = compute_mmd(np.array(feats_aneu), np.array(feats_card))
    else:
        mmd_cross_label[model_name] = np.nan

print("MMD cardiomegaly vs aortic enlargement, by model:")
for model_name, v in mmd_cross_label.items():
    print(f"  {model_name}: {v}")

df_metrics['MMD_pathology_vs_control'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('pathology_vs_control', np.nan), axis=1)
df_metrics['MMD_TP_vs_FP'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('TP_vs_FP', np.nan), axis=1)
df_metrics['MMD_TP_vs_FN'] = df_metrics.apply(
    lambda r: mmd_results.get((r['model'], r['label']), {}).get('TP_vs_FN', np.nan), axis=1)

UNIFIED_COLUMNS = [
    'image_id', 'model', 'label', 'resolution', 'iteracion',
    'patient_age', 'patient_age_group', 'patient_sex',
    'verdict', 'threshold_mcc', 'threshold_max_sens',
    'MCC', 'AUC_ROC', 'PR_AUC', 'sensitivity', 'specificity',
    'PG', 'PoE', 'DSC', 'IoU_between_maps', 'Del_AUC', 'Ins_AUC',
    'entropy_gradcam', 'MMD_pathology_vs_control', 'MMD_TP_vs_FP', 'MMD_TP_vs_FN',
]
df_metrics = df_metrics[UNIFIED_COLUMNS]
df_metrics.to_csv(f'{PROJECT_DIR}/xai_metrics.csv', index=False)
print(f"\nSaved unified results: {len(df_metrics)} rows to xai_metrics.csv")

In [ ]:
df = pd.read_csv(f'{PROJECT_DIR}/xai_metrics.csv')

print('\n=== Global metrics (per model, per label) ===')
global_agg = df.groupby(['model', 'label']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    IoU_between_maps=('IoU_between_maps', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(global_agg.to_string())

print('\n=== Stratified by verdict (TP/FP/FN) ===')
verdict_agg = df[df['verdict'] != 'TN'].groupby(['model', 'label', 'verdict']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(verdict_agg.to_string())

print('\n=== Stratified by PatientSex ===')
sex_agg = df.groupby(['model', 'label', 'patient_sex']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(sex_agg.to_string())

print('\n=== Stratified by PatientAge group ===')
age_agg = df.groupby(['model', 'label', 'patient_age_group']).agg(
    PG=('PG', 'mean'), PoE=('PoE', 'mean'), DSC=('DSC', 'mean'),
    Del_AUC=('Del_AUC', 'mean'), Ins_AUC=('Ins_AUC', 'mean'),
    entropy_gradcam=('entropy_gradcam', 'mean'), n=('image_id', 'count')
).round(4)
print(age_agg.to_string())

print('\nReminder: entropy_gradcam comparisons must stay within-architecture (AN vs AN, DN vs DN).')

global_agg.to_csv(f'{PROJECT_DIR}/global_xai_summary.csv')
verdict_agg.to_csv(f'{PROJECT_DIR}/stratification_verdict_xai_summary.csv')
sex_agg.to_csv(f'{PROJECT_DIR}/stratification_sex_xai_summary.csv')
age_agg.to_csv(f'{PROJECT_DIR}/stratification_age_xai_summary.csv')
print('\nTables saved')

In [ ]:
## IoU between maps of both labels

In [ ]:
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')

df_preds['type_aneurysm']     = df_preds.apply(lambda r: get_stratification(r, 'aneurysm'), axis=1)
df_preds['type_cardiomegaly'] = df_preds.apply(lambda r: get_stratification(r, 'cardiomegaly'), axis=1)
df_preds['combo']              = df_preds['type_aneurysm'] + '/' + df_preds['type_cardiomegaly']

results = []

for model in MODELS:
    print(f'Processing {model}...')
    df_m = df_preds[df_preds['model'] == model]

    for _, row in df_m.iterrows():
        image_id = row['image_id']
        combo    = row['combo']

        path = os.path.join(MAPS_DIR, model, f'{image_id}.npz')
        if not os.path.exists(path):
            continue

        data    = np.load(path)
        mapa_an = data['gradcam_aneurysm']
        mapa_ca = data['gradcam_cardiomegaly']

        iou = iou_between_maps(mapa_an, mapa_ca)

        results.append({
            'model':   model,
            'image_id': image_id,
            'combo':    combo,
            'iou':      iou
        })

df_iou = pd.DataFrame(results)

table = df_iou.groupby(['model', 'combo'])['iou'].agg(['mean', 'count']).round(4)
table.columns = ['IoU medio', 'n']
print('\nAverage IoU between maps by model and verdict combo:\n')
print(table.to_string())

df_iou.to_csv(f'{PROJECT_DIR}/iou_stratified_combo.csv', index=False)
table.to_csv(f'{PROJECT_DIR}/iou_stratified_summary.csv', index=False)
print('\nSaved.')

## Restore

In [ ]:
print(f'Removing {DATASET_DIR} and {MAPS_DIR}...')
shutil.rmtree(DATASET_DIR)
shutil.rmtree(MAPS_DIR)